# full_aistudio Data Integrity Quality Check

This notebook checks only `data_create/full_aistudio/` recursively. Each code cell checks one defect family and reports how many files are affected. The final cell writes all eliminated filenames to a txt log and moves/deletes the defect files, depending on the configured mode.


In [32]:
from pathlib import Path
import json
import re
import shutil
from collections import Counter, defaultdict

DATA_ROOT = Path("data_create/full_aistudio")
LAW_DOC_PATH = Path("law_doc.json")
MAX_DETAILS = 50

# Final-cell elimination settings.
ELIMINATED_LOG_PATH = Path("qualitycheck_eliminated_files.txt")
QUARANTINE_DIR = Path("qualitycheck_eliminated_files")
DELETE_FILES = False  # False = move to quarantine; True = permanently delete.

# Required schema for full_aistudio files. Edit these lists if your schema changes.
REQUIRED_TOP_LEVEL_FIELDS = [
    "THONG_TIN_CHUNG",
    "NOI_DUNG_VU_AN",
    "NHAN_DINH_CUA_TOA_AN",
    "De_Nghi_Cua_Vien_Kiem_Sat",
    "PHAN_QUYET_CUA_TOA_SO_THAM",
]
REQUIRED_THONG_TIN_CHUNG_FIELDS = [
    "Ma_Ban_An",
    "Thong_Tin_Bi_Cao",
]
REQUIRED_DEFENDANT_FIELDS = [
    "Ho_Ten",
]
REQUIRED_PROSECUTOR_FIELDS = ["Bi_Cao", "Pham_Toi", "Phat_Tu"]
REQUIRED_JUDGMENT_FIELDS = [
    "Bi_Cao",
    "Can_Cu_Dieu_Luat",
    "Pham_Toi",
    "Phat_Tu",
]

# Empty values are only considered defects for fields that should always contain content.
# Fields such as Tang_nang/Giam_nhe may be empty when no circumstance applies.
NON_EMPTY_FIELD_SUFFIXES = {
    "THONG_TIN_CHUNG.Ma_Ban_An",
    "THONG_TIN_CHUNG.Thong_Tin_Bi_Cao",
    "NOI_DUNG_VU_AN",
    "NHAN_DINH_CUA_TOA_AN",
    "De_Nghi_Cua_Vien_Kiem_Sat",
    "PHAN_QUYET_CUA_TOA_SO_THAM",
    "Ho_Ten",
    "Bi_Cao",
    "Pham_Toi",
    "Phat_Tu",
    "Can_Cu_Dieu_Luat",
}


JSON_FILES = sorted(path for path in DATA_ROOT.rglob("*.json") if path.is_file() and path.name != "law_doc.json")
ALL_SCAN_FILES = JSON_FILES

# Shared registries. Each check cell adds affected files here.
defect_files = defaultdict(set)
defect_details = defaultdict(list)
loaded_data = {}
valid_json_files = []


def rel_path(path):
    return Path(path).as_posix()


def register_defect(defect_name, file_paths, details=None):
    paths = {Path(path) for path in file_paths}
    defect_files[defect_name].update(paths)
    if details:
        defect_details[defect_name].extend(details)


def print_defect_report(title, affected_files, details=None):
    affected_files = sorted({Path(path) for path in affected_files}, key=rel_path)
    details = details or []
    total = len(ALL_SCAN_FILES)
    rate = (len(affected_files) / total * 100) if total else 0

    print("=" * 80)
    print(title)
    print("=" * 80)
    print(f"Folder scanned: {DATA_ROOT}")
    print(f"Total files scanned: {total}")
    print(f"Files affected: {len(affected_files)} ({rate:.2f}%)")

    if affected_files:
        print("\nSample affected files:")
        for path in affected_files[:MAX_DETAILS]:
            print(f"- {rel_path(path)}")
        if len(affected_files) > MAX_DETAILS:
            print(f"... and {len(affected_files) - MAX_DETAILS} more files")

    if details:
        print("\nSample details:")
        for item in details[:MAX_DETAILS]:
            print(item)
        if len(details) > MAX_DETAILS:
            print(f"... and {len(details) - MAX_DETAILS} more detail rows")


def is_missing_value(value):
    if value is None:
        return True
    if isinstance(value, str) and not value.strip():
        return True
    if isinstance(value, (list, dict)) and len(value) == 0:
        return True
    return False


def normalize_name(name):
    return " ".join(str(name or "").lower().split())


def get_name_sets(data):
    sources = {}

    thong_tin = data.get("THONG_TIN_CHUNG", {})
    if isinstance(thong_tin, dict):
        defendants = thong_tin.get("Thong_Tin_Bi_Cao", [])
        if isinstance(defendants, list):
            names = {
                normalize_name(item.get("Ho_Ten", ""))
                for item in defendants
                if isinstance(item, dict) and normalize_name(item.get("Ho_Ten", ""))
            }
            if names:
                sources["Thong_Tin_Bi_Cao"] = names

    for key in ("De_Nghi_Cua_Vien_Kiem_Sat", "PHAN_QUYET_CUA_TOA_SO_THAM"):
        items = data.get(key, [])
        if isinstance(items, list):
            names = {
                normalize_name(item.get("Bi_Cao", ""))
                for item in items
                if isinstance(item, dict) and normalize_name(item.get("Bi_Cao", ""))
            }
            if names:
                sources[key] = names

    return sources


def get_name_lists(data):
    sources = {}

    thong_tin = data.get("THONG_TIN_CHUNG", {})
    if isinstance(thong_tin, dict):
        defendants = thong_tin.get("Thong_Tin_Bi_Cao", [])
        if isinstance(defendants, list):
            names = [
                normalize_name(item.get("Ho_Ten", ""))
                for item in defendants
                if isinstance(item, dict) and normalize_name(item.get("Ho_Ten", ""))
            ]
            if names:
                sources["Thong_Tin_Bi_Cao"] = names

    for key in ("De_Nghi_Cua_Vien_Kiem_Sat", "PHAN_QUYET_CUA_TOA_SO_THAM"):
        items = data.get(key, [])
        if isinstance(items, list):
            names = [
                normalize_name(item.get("Bi_Cao", ""))
                for item in items
                if isinstance(item, dict) and normalize_name(item.get("Bi_Cao", ""))
            ]
            if names:
                sources[key] = names

    return sources


def get_nhan_dinh_text(data):
    nhan_dinh = data.get("NHAN_DINH_CUA_TOA_AN", data.get("Nhan_dinh_cua_toa_an", {}))
    if isinstance(nhan_dinh, dict):
        return "\n".join(str(value) for value in nhan_dinh.values() if value)
    if isinstance(nhan_dinh, list):
        return "\n".join(str(value) for value in nhan_dinh if value)
    return str(nhan_dinh or "")


ARTICLE_PATTERN = re.compile(r"(?i)điều\s+(\d+[a-zA-Z]*)")


def extract_dieu_from_text(text):
    return ARTICLE_PATTERN.findall(str(text or ""))


def is_blhs_reference(rule):
    bo_luat = str(rule.get("Bo_Luat_Va_Van_Ban_Khac", "")).strip().lower()
    return "blhs" in bo_luat or "bộ luật hình sự" in bo_luat


def extract_dieu_from_case(data):
    dieus = []

    for item in data.get("PHAN_QUYET_CUA_TOA_SO_THAM", []) if isinstance(data.get("PHAN_QUYET_CUA_TOA_SO_THAM", []), list) else []:
        if not isinstance(item, dict):
            continue
        rules = item.get("Can_Cu_Dieu_Luat", [])
        for rule in rules if isinstance(rules, list) else []:
            if isinstance(rule, dict) and is_blhs_reference(rule):
                dieu = str(rule.get("Dieu", "")).strip()
                if dieu:
                    dieus.append(dieu)

    if dieus:
        return dieus

    text_parts = [data.get("NOI_DUNG_VU_AN", ""), get_nhan_dinh_text(data)]
    for item in data.get("PHAN_QUYET_CUA_TOA_SO_THAM", []) if isinstance(data.get("PHAN_QUYET_CUA_TOA_SO_THAM", []), list) else []:
        if isinstance(item, dict):
            text_parts.extend(str(value) for value in item.values() if value)
    return extract_dieu_from_text("\n".join(text_parts))


def load_valid_dieus(law_doc_path):
    valid_dieus = set()
    if not law_doc_path.exists():
        print(f"Missing law document: {law_doc_path}")
        return valid_dieus
    with law_doc_path.open(encoding="utf-8") as fh:
        law_data = json.load(fh)
    for chuong in law_data:
        if isinstance(chuong, dict) and chuong.get("type") == "Chương":
            for dieu in chuong.get("Điều", []):
                if isinstance(dieu, dict) and dieu.get("type") == "Điều":
                    index = str(dieu.get("index", "")).strip()
                    if index:
                        valid_dieus.add(index)
    return valid_dieus


print(f"Folder scanned: {DATA_ROOT}")
print(f"JSON files found: {len(JSON_FILES)}")


Folder scanned: data_create/full_aistudio
JSON files found: 3586


## Check 1: invalid JSON files

In [33]:
invalid_json_files = []
invalid_json_details = []
loaded_data = {}
valid_json_files = []

for path in JSON_FILES:
    try:
        with path.open(encoding="utf-8") as fh:
            loaded_data[path] = json.load(fh)
        valid_json_files.append(path)
    except json.JSONDecodeError as exc:
        invalid_json_files.append(path)
        invalid_json_details.append({"file": rel_path(path), "error": str(exc)})
    except OSError as exc:
        invalid_json_files.append(path)
        invalid_json_details.append({"file": rel_path(path), "error": str(exc)})

register_defect("invalid_json", invalid_json_files, invalid_json_details)
print_defect_report("CHECK 1 - INVALID JSON", invalid_json_files, invalid_json_details)


CHECK 1 - INVALID JSON
Folder scanned: data_create/full_aistudio
Total files scanned: 3586
Files affected: 0 (0.00%)


## Check 2: missing required fields

In [34]:
missing_field_files = set()
missing_field_details = []
_seen_missing_fields = set()


def add_missing(path, field_path, reason):
    key = (rel_path(path), field_path, reason)
    if key in _seen_missing_fields:
        return
    _seen_missing_fields.add(key)
    missing_field_files.add(path)
    missing_field_details.append({"file": rel_path(path), "field": field_path, "reason": reason})


def should_require_non_empty(field_path, field_name):
    return field_path in NON_EMPTY_FIELD_SUFFIXES or field_name in NON_EMPTY_FIELD_SUFFIXES


def check_required_dict_fields(path, obj, required_fields, prefix):
    if not isinstance(obj, dict):
        add_missing(path, prefix, f"expected dict, got {type(obj).__name__}")
        return
    for field in required_fields:
        field_path = f"{prefix}.{field}"
        if field not in obj:
            add_missing(path, field_path, "missing key")
        elif should_require_non_empty(field_path, field) and is_missing_value(obj[field]):
            add_missing(path, field_path, "empty value")


def check_required_list_item_fields(path, items, required_fields, prefix):
    if not isinstance(items, list):
        add_missing(path, prefix, f"expected list, got {type(items).__name__}")
        return
    for item in items:
        if not isinstance(item, dict):
            add_missing(path, prefix, f"contains non-dict item: {type(item).__name__}")
            continue
        for field in required_fields:
            field_path = f"{prefix}.{field}"
            if field not in item:
                add_missing(path, field_path, "missing key in one or more items")
            elif should_require_non_empty(field_path, field) and is_missing_value(item[field]):
                add_missing(path, field_path, "empty value in one or more items")


for path, data in loaded_data.items():
    check_required_dict_fields(path, data, REQUIRED_TOP_LEVEL_FIELDS, "root")

    thong_tin = data.get("THONG_TIN_CHUNG")
    check_required_dict_fields(path, thong_tin, REQUIRED_THONG_TIN_CHUNG_FIELDS, "THONG_TIN_CHUNG")

    defendants = thong_tin.get("Thong_Tin_Bi_Cao", []) if isinstance(thong_tin, dict) else []
    check_required_list_item_fields(path, defendants, REQUIRED_DEFENDANT_FIELDS, "THONG_TIN_CHUNG.Thong_Tin_Bi_Cao")

    prosecutor_items = data.get("De_Nghi_Cua_Vien_Kiem_Sat", [])
    check_required_list_item_fields(path, prosecutor_items, REQUIRED_PROSECUTOR_FIELDS, "De_Nghi_Cua_Vien_Kiem_Sat")

    judgment_items = data.get("PHAN_QUYET_CUA_TOA_SO_THAM", [])
    check_required_list_item_fields(path, judgment_items, REQUIRED_JUDGMENT_FIELDS, "PHAN_QUYET_CUA_TOA_SO_THAM")

register_defect("missing_required_fields", missing_field_files, missing_field_details)
print_defect_report("CHECK 2 - MISSING REQUIRED FIELDS", missing_field_files)

field_to_files = defaultdict(set)
field_to_missing_files = defaultdict(set)
field_to_empty_files = defaultdict(set)
file_to_missing_fields = defaultdict(list)

for item in missing_field_details:
    file_name = item["file"]
    field = item["field"]
    reason = item["reason"]
    field_to_files[field].add(file_name)
    if "missing" in reason:
        field_to_missing_files[field].add(file_name)
    if "empty" in reason:
        field_to_empty_files[field].add(file_name)
    file_to_missing_fields[file_name].append(f"{field} ({reason})")

field_rows = sorted(
    (
        field,
        len(field_to_missing_files[field]),
        len(field_to_empty_files[field]),
        len(field_to_files[field]),
    )
    for field in field_to_files
)

print("\nFIELD DEFECT FILE COUNTS:")
if field_rows:
    print(f"{'field':<65} {'missing_files':>13} {'empty_files':>11} {'total_files':>11}")
    print("-" * 104)
    for field, missing_count, empty_count, total_count in field_rows:
        print(f"{field:<65} {missing_count:>13} {empty_count:>11} {total_count:>11}")
else:
    print("No missing required fields found.")

print("\nFILES AND EXACT MISSING FIELDS:")
if file_to_missing_fields:
    for file_name in sorted(file_to_missing_fields)[:MAX_DETAILS]:
        fields = file_to_missing_fields[file_name]
        print(f"- {file_name}")
        for field in fields[:20]:
            print(f"  * {field}")
        if len(fields) > 20:
            print(f"  * ... and {len(fields) - 20} more missing fields in this file")
    if len(file_to_missing_fields) > MAX_DETAILS:
        print(f"... and {len(file_to_missing_fields) - MAX_DETAILS} more affected files")
else:
    print("No files have missing required fields.")


CHECK 2 - MISSING REQUIRED FIELDS
Folder scanned: data_create/full_aistudio
Total files scanned: 3586
Files affected: 268 (7.47%)

Sample affected files:
- data_create/full_aistudio/02-01-2025-Thanh_Hoa-2ta1733765t1cvn.json
- data_create/full_aistudio/03-01-2024-Lai_Chau-2ta1409450t1cvn.json
- data_create/full_aistudio/03-01-2025-Binh_Phuoc-2ta1753290t1cvn.json
- data_create/full_aistudio/03-04-2025-An_Giang-2ta1923855t1cvn.json
- data_create/full_aistudio/03-07-2024-Hai_Duong-2ta1545419t1cvn.json
- data_create/full_aistudio/04-04-2024-Ha_Tinh-2ta1550422t1cvn.json
- data_create/full_aistudio/04-07-2025-Hung_Yen-2ta1874873t1cvn.json
- data_create/full_aistudio/05-01-2024-Hai_Phong-2ta1415115t1cvn.json
- data_create/full_aistudio/05-01-2024-Nam_Dinh-2ta1411436t1cvn.json
- data_create/full_aistudio/05-02-2024-Hung_Yen-2ta1561043t1cvn.json
- data_create/full_aistudio/05-03-2025-Ninh_Binh-2ta1773623t1cvn.json
- data_create/full_aistudio/05-03-2025-Quang_Tri-2ta1909732t1cvn.json
- data_creat

## Check 3: compact defendant mismatch check

In [35]:
defendant_mismatch_files = []
defendant_mismatch_details = []

for path, data in loaded_data.items():
    names_by_source = get_name_sets(data)
    if len(names_by_source) <= 1:
        continue

    canonical_sets = {tuple(sorted(names)) for names in names_by_source.values()}
    if len(canonical_sets) > 1:
        defendant_mismatch_files.append(path)
        defendant_mismatch_details.append({
            "file": rel_path(path),
            "counts": {source: len(names) for source, names in names_by_source.items()},
            "names": {source: sorted(names) for source, names in names_by_source.items()},
        })

register_defect("defendant_mismatch", defendant_mismatch_files, defendant_mismatch_details)
print_defect_report("CHECK 3 - DEFENDANT MISMATCH", defendant_mismatch_files, defendant_mismatch_details)


CHECK 3 - DEFENDANT MISMATCH
Folder scanned: data_create/full_aistudio
Total files scanned: 3586
Files affected: 99 (2.76%)

Sample affected files:
- data_create/full_aistudio/02-01-2025-Quang_Ninh-2ta1714292t1cvn.json
- data_create/full_aistudio/02-02-2024-Binh_Phuoc-2ta1436434t1cvn.json
- data_create/full_aistudio/02-08-2024-Ho_Chi_Minh-2ta1629952t1cvn.json
- data_create/full_aistudio/02-08-2024-Ho_Chi_Minh-2ta1960999t1cvn.json
- data_create/full_aistudio/04-07-2024-Lai_Chau-2ta1561304t1cvn.json
- data_create/full_aistudio/04-07-2024-Quang_Ninh-2ta1548186t1cvn.json
- data_create/full_aistudio/05-03-2025-Tien_Giang-2ta1819516t1cvn.json
- data_create/full_aistudio/05-04-2024-Ho_Chi_Minh-2ta1644132t1cvn.json
- data_create/full_aistudio/06-01-2025-Quang_Ninh-2ta1735836t1cvn.json
- data_create/full_aistudio/06-08-2025-Dak_Lak-2ta1919024t1cvn.json
- data_create/full_aistudio/06-11-2025-Ho_Chi_Minh-2ta2078511t1cvn.json
- data_create/full_aistudio/07-03-2025-Ha_Noi-2ta1949984t1cvn.json
- dat

## Check 4: duplicate defendants within one section

In [36]:
duplicate_defendant_files = set()
duplicate_defendant_details = []

for path, data in loaded_data.items():
    for source, names in get_name_lists(data).items():
        duplicates = {name: count for name, count in Counter(names).items() if count > 1}
        if duplicates:
            duplicate_defendant_files.add(path)
            duplicate_defendant_details.append({
                "file": rel_path(path),
                "source": source,
                "duplicates": duplicates,
            })

register_defect("duplicate_defendants", duplicate_defendant_files, duplicate_defendant_details)
print_defect_report("CHECK 4 - DUPLICATE DEFENDANTS", duplicate_defendant_files, duplicate_defendant_details)


CHECK 4 - DUPLICATE DEFENDANTS
Folder scanned: data_create/full_aistudio
Total files scanned: 3586
Files affected: 5 (0.14%)

Sample affected files:
- data_create/full_aistudio/03-01-2024-Lai_Chau-2ta1409450t1cvn.json
- data_create/full_aistudio/05-04-2024-Vinh_Long-2ta1502346t1cvn.json
- data_create/full_aistudio/20-05-2025-Lao_Cai-2ta1963916t1cvn.json
- data_create/full_aistudio/20-06-2024-Bac_Giang-2ta1523335t1cvn.json
- data_create/full_aistudio/23-01-2025-Khanh_Hoa-2ta1714295t1cvn.json

Sample details:
{'file': 'data_create/full_aistudio/03-01-2024-Lai_Chau-2ta1409450t1cvn.json', 'source': 'PHAN_QUYET_CUA_TOA_SO_THAM', 'duplicates': {'chảo a p': 2}}
{'file': 'data_create/full_aistudio/05-04-2024-Vinh_Long-2ta1502346t1cvn.json', 'source': 'PHAN_QUYET_CUA_TOA_SO_THAM', 'duplicates': {'doàn công m': 2}}
{'file': 'data_create/full_aistudio/20-05-2025-Lao_Cai-2ta1963916t1cvn.json', 'source': 'PHAN_QUYET_CUA_TOA_SO_THAM', 'duplicates': {'tri\u1c3a1u th\u1c3al t': 2}}
{'file': 'data_crea

## Check 5: missing legal article references

In [37]:
missing_article_files = []
missing_article_details = []

for path, data in loaded_data.items():
    dieus = set(extract_dieu_from_case(data))
    if not dieus:
        missing_article_files.append(path)
        missing_article_details.append({"file": rel_path(path), "defect": "No Điều references extracted"})

register_defect("missing_article_references", missing_article_files, missing_article_details)
print_defect_report("CHECK 5 - MISSING LEGAL ARTICLE REFERENCES", missing_article_files, missing_article_details)


CHECK 5 - MISSING LEGAL ARTICLE REFERENCES
Folder scanned: data_create/full_aistudio
Total files scanned: 3586
Files affected: 0 (0.00%)


## Check 6: invalid legal article references against law_doc.json

In [38]:
valid_dieus = load_valid_dieus(LAW_DOC_PATH)
invalid_article_files = set()
invalid_article_details = []
total_article_references_checked = 0

for path, data in loaded_data.items():
    for dieu in sorted(set(extract_dieu_from_case(data))):
        total_article_references_checked += 1
        if valid_dieus and dieu not in valid_dieus:
            invalid_article_files.add(path)
            invalid_article_details.append({
                "file": rel_path(path),
                "dieu": dieu,
                "error": f"Điều '{dieu}' not found in {LAW_DOC_PATH}",
            })

register_defect("invalid_article_references", invalid_article_files, invalid_article_details)
print_defect_report("CHECK 6 - INVALID LEGAL ARTICLE REFERENCES", invalid_article_files, invalid_article_details)
print(f"Unique per-file article references checked: {total_article_references_checked}")
print(f"Valid Điều loaded: {len(valid_dieus)}")


CHECK 6 - INVALID LEGAL ARTICLE REFERENCES
Folder scanned: data_create/full_aistudio
Total files scanned: 3586
Files affected: 7 (0.20%)

Sample affected files:
- data_create/full_aistudio/04-07-2025-Hung_Yen-2ta1874873t1cvn.json
- data_create/full_aistudio/06-03-2025-Hai_Phong-2ta1833109t1cvn.json
- data_create/full_aistudio/21-01-2025-Da_Nang-2ta1764624t1cvn.json
- data_create/full_aistudio/21-01-2025-Nghe_An-2ta1969164t1cvn.json
- data_create/full_aistudio/23-12-2025-Dien_Bien-2ta2028372t1cvn.json
- data_create/full_aistudio/25-03-2025-Ha_Noi-2ta1928305t1cvn.json
- data_create/full_aistudio/25-11-2025-Quang_Tri-2ta2010627t1cvn.json

Sample details:
{'file': 'data_create/full_aistudio/04-07-2025-Hung_Yen-2ta1874873t1cvn.json', 'dieu': '317', 'error': "Điều '317' not found in law_doc.json"}
{'file': 'data_create/full_aistudio/06-03-2025-Hai_Phong-2ta1833109t1cvn.json', 'dieu': '말투', 'error': "Điều '말투' not found in law_doc.json"}
{'file': 'data_create/full_aistudio/21-01-2025-Da_Nang-

## Final cell: eliminate all defect files and save eliminated filenames

In [39]:
all_bad_files = sorted(set().union(*defect_files.values()) if defect_files else set(), key=rel_path)
file_to_defects = defaultdict(list)
for defect_name, paths in defect_files.items():
    for path in paths:
        file_to_defects[path].append(defect_name)

print("=" * 80)
print("FINAL ELIMINATION")
print("=" * 80)
print(f"Total defect files to eliminate: {len(all_bad_files)}")
print(f"Mode: {'DELETE' if DELETE_FILES else 'MOVE TO QUARANTINE'}")
print(f"Log path: {ELIMINATED_LOG_PATH}")
if not DELETE_FILES:
    print(f"Quarantine folder: {QUARANTINE_DIR}")

log_lines = [
    "# Eliminated defect files",
    f"source_root={DATA_ROOT}",
    f"mode={'delete' if DELETE_FILES else 'move_to_quarantine'}",
    f"total={len(all_bad_files)}",
    "",
]

eliminated_count = 0
missing_at_elimination = []

for path in all_bad_files:
    defects = ",".join(sorted(file_to_defects[path]))
    log_lines.append(f"{rel_path(path)}\t{defects}")

    if not path.exists():
        missing_at_elimination.append(path)
        continue

    if DELETE_FILES:
        path.unlink()
    else:
        target = QUARANTINE_DIR / path
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            stem = target.stem
            suffix = target.suffix
            counter = 1
            while target.exists():
                target = target.with_name(f"{stem}.{counter}{suffix}")
                counter += 1
        shutil.move(str(path), str(target))
    eliminated_count += 1

ELIMINATED_LOG_PATH.write_text("\n".join(log_lines) + "\n", encoding="utf-8")

print(f"Eliminated files: {eliminated_count}")
print(f"Missing before elimination: {len(missing_at_elimination)}")
print(f"Saved eliminated filename log to: {ELIMINATED_LOG_PATH}")

if all_bad_files:
    print("\nSample eliminated files:")
    for path in all_bad_files[:MAX_DETAILS]:
        print(f"- {rel_path(path)} [{', '.join(sorted(file_to_defects[path]))}]")
    if len(all_bad_files) > MAX_DETAILS:
        print(f"... and {len(all_bad_files) - MAX_DETAILS} more files")
else:
    print("No defect files were found by the checks above.")


FINAL ELIMINATION
Total defect files to eliminate: 363
Mode: MOVE TO QUARANTINE
Log path: qualitycheck_eliminated_files.txt
Quarantine folder: qualitycheck_eliminated_files
Eliminated files: 363
Missing before elimination: 0
Saved eliminated filename log to: qualitycheck_eliminated_files.txt

Sample eliminated files:
- data_create/full_aistudio/02-01-2025-Quang_Ninh-2ta1714292t1cvn.json [defendant_mismatch]
- data_create/full_aistudio/02-01-2025-Thanh_Hoa-2ta1733765t1cvn.json [missing_required_fields]
- data_create/full_aistudio/02-02-2024-Binh_Phuoc-2ta1436434t1cvn.json [defendant_mismatch]
- data_create/full_aistudio/02-08-2024-Ho_Chi_Minh-2ta1629952t1cvn.json [defendant_mismatch]
- data_create/full_aistudio/02-08-2024-Ho_Chi_Minh-2ta1960999t1cvn.json [defendant_mismatch]
- data_create/full_aistudio/03-01-2024-Lai_Chau-2ta1409450t1cvn.json [duplicate_defendants, missing_required_fields]
- data_create/full_aistudio/03-01-2025-Binh_Phuoc-2ta1753290t1cvn.json [missing_required_fields]
-